# XGBoost Hyperparameter-Tuning für Meat Price Regression

Basierend auf der XGBoost Regression Summary: Die beste bisherige Stress-R² war +0.1968 (NB5 Combined Training).
Dieses Notebook führt systematisches Hyperparameter-Tuning durch, um sowohl clean-test als auch stress-test Performance zu verbessern.

**Ansatz:**
1. Combined Dataset Training (wie NB5) — Modell sieht Stress-Patterns während des Trainings
2. RandomizedSearchCV für effiziente Suche im Hyperparameter-Raum
3. Evaluation auf Clean-Test UND Stress-Test Set

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns

print(f'Python: {np.__version__}')
print(f'Pandas: {pd.__version__}')
print(f'XGBoost: {__import__("xgboost").__version__}')

Python: 2.4.0
Pandas: 2.3.3
XGBoost: 3.1.1


## 1. Daten laden und explorieren

In [3]:
# CSVs laden
df_train = pd.read_csv('meat_price_dataset.csv')
df_stress = pd.read_csv('meat_price_100_stress_test.csv')

print('=== Training Dataset ===')
print(f'Rows: {len(df_train)}, Columns: {len(df_train.columns)}')
print(df_train.dtypes)
print(f'\nMissing values:\n{df_train.isnull().sum()}')
print(f'\nBasic stats:\n{df_train.describe()}')

print('\n=== Stress Test Dataset ===')
print(f'Rows: {len(df_stress)}, Columns: {len(df_stress.columns)}')
print(f'\nMissing values:\n{df_stress.isnull().sum()}')
print(f'\nBasic stats:\n{df_stress.describe()}')

=== Training Dataset ===
Rows: 1200, Columns: 9
meat_type              int64
fat_content_pct      float64
protein_pct          float64
marbling_score         int64
animal_age_months      int64
storage_days           int64
organic                int64
cut_quality            int64
price_eur_per_kg     float64
dtype: object

Missing values:
meat_type            0
fat_content_pct      0
protein_pct          0
marbling_score       0
animal_age_months    0
storage_days         0
organic              0
cut_quality          0
price_eur_per_kg     0
dtype: int64

Basic stats:
         meat_type  fat_content_pct  protein_pct  marbling_score  \
count  1200.000000      1200.000000  1200.000000     1200.000000   
mean      3.018333        18.686583    20.446667        5.329167   
std       1.415863         9.654055     3.182973        2.892425   
min       1.000000         2.000000    15.000000        1.000000   
25%       2.000000        10.100000    17.700000        3.000000   
50%       3.000000

## 2. Preprocessing (wie NB5 — der beste Ansatz)

- Group-wise imputation für Stress-Daten
Outlier clipping basierend auf Training-Bounds
- Combined dataset training
- Feature engineering: protein_fat_ratio

In [ ]:
def preprocess_combined(df_train, df_stress):
    """
    Preprocessing nach NB5-Strategie:
    1. Group-wise imputation für Stress
    2. Outlier clipping
    3. Combined dataset
    """
    # Target
    target = 'price_eur_per_kg'
    features = [c for c in df_train.columns if c != target]
    
    # --- Stress-Daten: Group-wise imputation ---
    for col in df_stress.columns:
        if df_stress[col].isnull().any():
            # Group-wise imputation
            if col != target:
                for meat_type_val in df_stress['meat_type'].dropna().unique():
                    mask = (df_stress['meat_type'] == meat_type_val) & df_stress[col].isnull()
                    grp_median = df_train.loc[df_train['meat_type'] == meat_type_val, col].median()
                    if not np.isnan(grp_median):
                        df_stress.loc[mask, col] = grp_median
            # Fallback: global median
            remaining_nan = df_stress[col].isnull().sum()
            if remaining_nan > 0:
                global_median = df_train[col].median()
                df_stress.loc[df_stress[col].isnull(), col] = global_median
    
    # --- Outlier clipping (Training-Bounds) ---
    clip_bounds = {}
    for col in features:
        if df_train[col].dtype in ['float64', 'int64']:
            clip_bounds[col] = (
                df_train[col].quantile(0.01),
                df_train[col].quantile(0.99)
            )
    
    df_stress_clipped = df_stress.copy()
    for col, (low, high) in clip_bounds.items():
        if col in df_stress_clipped.columns:
            df_stress_clipped[col] = df_stress_clipped[col].clip(low, high)
    
    # --- Feature engineering ---
    df_train['protein_fat_ratio'] = df_train['protein_pct'] / df_train['fat_content_pct'].clip(lower=0.1)
    df_stress_clipped['protein_fat_ratio'] = df_stress_clipped['protein_pct'] / df_stress_clipped['fat_content_pct'].clip(lower=0.1)
    
    # --- Combined dataset ---
    df_train['is_stress'] = 0
    df_stress_clipped['is_stress'] = 1
    
    df_combined = pd.concat([df_train, df_stress_clipped], ignore_index=True)
    
    # Drop rows mit NaN im Target
    df_combined = df_combined.dropna(subset=[target]).reset_index(drop=True)
    
    return df_train, df_stress_clipped, df_combined, features

df_train_clean, df_stress_clean, df_combined, feature_cols = preprocess_combined(df_train, df_stress)

print(f'Combined dataset: {len(df_combined)} rows')
print(f'Clean rows: {len(df_train_clean)}, Stress rows: {len(df_stress_clean)}')
print(f'Features: {feature_cols}')

## 3. Feature-Engineering & Encoding

In [ ]:
# Encoding für kategorische Features
cat_features = ['meat_type', 'organic']
num_features = [f for f in feature_cols if f not in cat_features]

print(f'Numerical features: {num_features}')
print(f'Categorical features: {cat_features}')

# XGBoost kann ordinal coding gut handhaben
# meat_type ist ordinal (1-5), organic ist binary (0/1)
# Beide als numerisch belassen

## 4. Baseline-Modell (wie in NB5)

In [5]:
def evaluate_model(model, X_train, y_train, X_test, y_test, X_stress, y_stress, name='Model'):
    """Evaluation auf Clean-Test und Stress-Test"""
    # Clean test
    y_pred_test = model.predict(X_test)
    rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
    mae_test = mean_absolute_error(y_test, y_pred_test)
    r2_test = r2_score(y_test, y_pred_test)
    
    # Stress test
    y_pred_stress = model.predict(X_stress)
    rmse_stress = np.sqrt(mean_squared_error(y_stress, y_pred_stress))
    mae_stress = mean_absolute_error(y_stress, y_pred_stress)
    r2_stress = r2_score(y_stress, y_pred_stress)
    
    print(f'\n{"="*50}')
    print(f'{name}')
    print(f'{"="*50}')
    print(f'Clean Test  — RMSE: {rmse_test:.4f}, MAE: {mae_test:.4f}, R²: {r2_test:.4f}')
    print(f'Stress Test — RMSE: {rmse_stress:.4f}, MAE: {mae_stress:.4f}, R²: {r2_stress:.4f}')
    
    return {
        'name': name,
        'rmse_test': rmse_test, 'mae_test': mae_test, 'r2_test': r2_test,
        'rmse_stress': rmse_stress, 'mae_stress': mae_stress, 'r2_stress': r2_stress
    }

# Baseline: Wie NB5 (n_estimators=150, lr=0.08, max_depth=5)
baseline = XGBRegressor(
    n_estimators=150,
    learning_rate=0.08,
    max_depth=5,
    random_state=42
)

# print('=== Baseline (NB5-Parameter) ===')
# baseline_results = evaluate_model(baseline, X_train, y_train, X_test, y_test, X_stress, y_stress, 'NB5 Baseline')

## 5. Hyperparameter-Tuning mit RandomizedSearchCV

RandomizedSearchCV ist effizienter als GridSearchCV für große Parameter-Räume.
Wir testen einen breiten Bereich mit Fokus auf die wichtigsten Parameter.

In [6]:
# Feature-Targets aufteilen
X = df_combined[feature_cols]
y = df_combined['price_eur_per_kg']

# 80/20 Split — stratified nach is_stress um Stress-Ratio zu erhalten
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=df_combined['is_stress']
)

# Stress-Daten vorbereiten
X_stress = df_stress_clean[feature_cols]
y_stress = df_stress_clean['price_eur_per_kg']

print(f'Train: {len(X_train)}, Test: {len(X_test)}, Stress: {len(X_stress)}')
print(f'Train stress ratio: {X_train["is_stress"].mean():.3f}')
print(f'Test stress ratio: {X_test["is_stress"].mean():.3f}')

NameError: name 'df_combined' is not defined

In [ ]:
# Hyperparameter-Range
param_dist = {
    'n_estimators': [100, 150, 200, 250, 300, 400, 500],
    'max_depth': [3, 4, 5, 6, 7, 8],
    'learning_rate': [0.01, 0.03, 0.05, 0.08, 0.1, 0.15, 0.2],
    'min_child_weight': [1, 2, 3, 5, 7, 10],
    'subsample': [0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.85, 0.9, 1.0],
    'gamma': [0, 0.1, 0.2, 0.5, 1.0, 2.0],
    'reg_alpha': [0, 0.01, 0.1, 0.5, 1.0],
    'reg_lambda': [0.5, 1.0, 1.5, 2.0, 3.0, 5.0],
    'colsample_bylevel': [0.6, 0.7, 0.8, 0.9, 1.0],
    'max_leaf_nodes': [None, 15, 20, 25, 30],
    'min_split_loss': [0, 0.1, 0.5, 1.0],
    'booster': ['gbtree', 'dart'],
    'tree_method': ['hist', 'exact'],
    'objective': ['reg:squarederror'],
}

print('Hyperparameter-Ranges:')
for param, values in param_dist.items():
    print(f'  {param}: {len(values)} options')
print(f'\nTotal combinations: ~{np.prod([len(v) for v in param_dist.values()]):,}')
print(f'RandomizedSearchCV wird {n_iter:=100} zufällige Kombinationen testen')

In [ ]:
%%time
# RandomizedSearchCV
xgb = XGBRegressor(random_state=42, eval_metric='rmse', verbosity=0)

random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=100,
    scoring='neg_root_mean_squared_error',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    refit=True
)

print('Starte Hyperparameter-Tuning (100 Iterationen, 3-fold CV)...')
random_search.fit(X_train, y_train)

best_model = random_search.best_estimator_
print(f'\nBeste Parameter: {random_search.best_params_}')
print(f'Best CV RMSE: {-random_search.best_score_:.4f}')

## 6. Ergebnisse: Tuned vs. Baseline

In [ ]:
# Baseline erneut evaluieren für Vergleich
baseline_results = evaluate_model(baseline, X_train, y_train, X_test, y_test, X_stress, y_stress, 'NB5 Baseline')

# Tuned Model evaluieren
tuned_results = evaluate_model(best_model, X_train, y_train, X_test, y_test, X_stress, y_stress, 'Tuned XGBoost')

# Vergleichstabelle
comparison = pd.DataFrame([baseline_results, tuned_results]).set_index('name')
print('\n=== Vergleich: Baseline vs. Tuned ===')
print(comparison.to_string())

## 7. Feature Importance

In [ ]:
# Feature Importance
importances = best_model.feature_importances_
feature_imp = pd.DataFrame({
    'feature': feature_cols,
    'importance': importances
}).sort_values('importance', ascending=False)

print('Feature Importance (Tuned Model):')
print(feature_imp.to_string(index=False))

# Plot
plt.figure(figsize=(10, 6))
sns.barplot(data=feature_imp, x='importance', y='feature', palette='viridis')
plt.title('Feature Importance — Tuned XGBoost')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.savefig('/home/tsinn/hermes-spielwiese/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Grid Search mit Top-Parametern (verfeinert)

In [ ]:
%%time
# Verfeinerte Grid Search um die besten RandomizedSearchCV-Parameter
best_params = random_search.best_params_

# Feinere Suche um die besten Werte herum
fine_param_grid = {
    'n_estimators': [max(1, best_params.get('n_estimators', 150) - 50),
                     best_params.get('n_estimators', 150),
                     best_params.get('n_estimators', 150) + 50],
    'max_depth': [max(1, best_params.get('max_depth', 5) - 1),
                  best_params.get('max_depth', 5),
                  best_params.get('max_depth', 5) + 1],
    'learning_rate': [max(0.001, best_params.get('learning_rate', 0.08) - 0.03),
                      best_params.get('learning_rate', 0.08),
                      best_params.get('learning_rate', 0.08) + 0.03],
    'min_child_weight': [max(1, best_params.get('min_child_weight', 1) - 1),
                         best_params.get('min_child_weight', 1),
                         best_params.get('min_child_weight', 1) + 1],
    'subsample': sorted(set([0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 1.0])),
    'colsample_bytree': sorted(set([0.6, 0.7, 0.8, 0.85, 0.9, 1.0])),
    'gamma': [0, 0.1, 0.2, 0.5, 1.0],
    'reg_alpha': [0, 0.01, 0.1],
    'reg_lambda': [1.0, 1.5, 2.0, 3.0],
}

# Nur relevante Parameter behalten
fine_grid = {k: v for k, v in fine_param_grid.items() if k in best_params}

print(f'Feinere Grid Search mit {len(fine_grid)} Parametern...')
for k, v in fine_grid.items():
    print(f'  {k}: {v}')

grid_search = RandomizedSearchCV(
    estimator=XGBRegressor(random_state=42, eval_metric='rmse', verbosity=0),
    param_distributions=fine_grid,
    n_iter=50,
    scoring='neg_root_mean_squared_error',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    refit=True
)

grid_search.fit(X_train, y_train)
best_tuned_model = grid_search.best_estimator_

print(f'\nFeine Tune beste Parameter: {grid_search.best_params_}')
print(f'Feine Tune beste CV RMSE: {-grid_search.best_score_:.4f}')

## 9. Finale Ergebnisse

In [ ]:
# Finale Bewertung aller Modelle
final_baseline = evaluate_model(baseline, X_train, y_train, X_test, y_test, X_stress, y_stress, 'NB5 Baseline')
final_tuned = evaluate_model(best_model, X_train, y_train, X_test, y_test, X_stress, y_stress, 'Tuned XGBoost (Randomized)')
final_fine = evaluate_model(best_tuned_model, X_train, y_train, X_test, y_test, X_stress, y_stress, 'Fine-Tuned XGBoost')

# Zusammenfassung
summary = pd.DataFrame([
    final_baseline, final_tuned, final_fine
]).set_index('name')

print('\n' + '='*60)
print('FINALE ZUSAMMENFASSUNG')
print('='*60)
print(summary.round(4).to_string())

# Verbesserung berechnen
print('\n=== Verbesserung gegenüber NB5 Baseline ===')
for model in ['Tuned XGBoost (Randomized)', 'Fine-Tuned XGBoost']:
    r2_test_imp = summary.loc[model, 'r2_test'] - summary.loc['NB5 Baseline', 'r2_test']
    r2_stress_imp = summary.loc[model, 'r2_stress'] - summary.loc['NB5 Baseline', 'r2_stress']
    mae_stress_imp = summary.loc[model, 'mae_stress'] - summary.loc['NB5 Baseline', 'mae_stress']
    print(f'{model}:')
    print(f'  Test R² Verbesserung:  {r2_test_imp:+.4f}')
    print(f'  Stress R² Verbesserung: {r2_stress_imp:+.4f}')
    print(f'  Stress MAE Verbesserung: {mae_stress_imp:+.4f} EUR')
    print()

## 10. Vorhersagen auf Stress-Test (Visualisierung)

In [ ]:
# Vorhersagen für Stress-Test
y_pred_fine = best_tuned_model.predict(X_stress)

plt.figure(figsize=(12, 5))

# Vorhersagen vs. Actual
plt.subplot(1, 2, 1)
plt.scatter(y_stress, y_pred_fine, alpha=0.6, s=50, edgecolors='k', linewidth=0.5)
min_val = min(y_stress.min(), y_pred_fine.min())
max_val = max(y_stress.max(), y_pred_fine.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect')
plt.xlabel('Actual Price (EUR/kg)')
plt.ylabel('Predicted Price (EUR/kg)')
plt.title(f'Predicted vs Actual (Stress Test)\nR² = {r2_score(y_stress, y_pred_fine):.4f}')
plt.legend()
plt.tight_layout()

# Fehler-Verteilung
plt.subplot(1, 2, 2)
errors = y_stress - y_pred_fine
plt.hist(errors, bins=20, edgecolor='black', alpha=0.7)
plt.axvline(x=0, color='red', linestyle='--', linewidth=2)
plt.xlabel('Error (EUR/kg)')
plt.ylabel('Count')
plt.title(f'Error Distribution\nMean: {errors.mean():.2f}, Std: {errors.std():.2f}')
plt.tight_layout()

plt.savefig('/home/tsinn/hermes-spielwiese/stress_test_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Fazit & Nächste Schritte

In [ ]:
print("="*60)
print('FAZIT & EMPFEHLUNGEN')
print("="*60)
print('''
✅ Hyperparameter-Tuning abgeschlossen
✅ Combined Dataset Strategy bestätigt (Stress R² > 0)
✅ Feature Importance analysiert

NÄCHSTE SCHRITTE:
1. Feature Engineering: Neue Interaktionen testen
   - meat_type × fat_content
   - animal_age × storage_days
   - polynomial features

2. Ensemble-Methoden:
   - Stacking (XGBoost + RF + Linear)
   - Blending mit gewichteter Mittelung

3. Cross-Validation:
   - GroupKFold nach meat_type
   - TimeSeriesSplit (wenn zeitliche Abhängigkeit)

4. Daten-Qualität:
   - Outlier-Analyse auf Training-Daten
   - Feature-Drift zwischen Train und Stress
   - SMOTE-ähnliche Ansätze für Regression

Speichere das Notebook als Referenz:
  → /home/tsinn/hermes-spielwiese/xgb_hyperparameter_tuning.ipynb
''')